# 5-Fold Stratified Cross-Validation Analizi

Bu çalışmada, siroz hastalarının durum tahmini veri seti üzerinde eğitilen final model adayımızın farklı veri bölmelerinde ne kadar stabil çalıştığı ve genellenebilirliği **5-Fold Stratified Cross-Validation** kullanılarak doğrulanmıştır.

### Model Tanımı:
- **Model:** `LGBMClassifier`
- **class_weight:** `'balanced'`
- **Parametreler:** Hiperparametre optimizasyonu sonucu elde edilen en iyi parametreler.

### Değerlendirilen Metrikler:
Her fold için eğitim (train) ve doğrulama (val) setlerinde aşağıdaki metrikler hesaplanmıştır:
- Doğruluk (Accuracy)
- Macro F1 Score
- Precision Macro
- Recall Macro
- Balanced Accuracy


In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score
from lightgbm import LGBMClassifier

print("Kütüphaneler başarıyla yüklendi.")

Kütüphaneler başarıyla yüklendi.


In [2]:
# Veri Yükleme
df = pd.read_csv('../data/processed_train.csv')
X = df.drop(columns=['Status'])
y = df['Status']

print(f"Toplam Veri Seti Boyutu: {X.shape}")

Toplam Veri Seti Boyutu: (7905, 30)


In [3]:
# En iyi parametreler ve model tanımı
best_params = {
    'subsample_freq': 1,
    'subsample': 1.0,
    'reg_lambda': 0.1,
    'reg_alpha': 5.0,
    'num_leaves': 127,
    'n_estimators': 500,
    'min_child_samples': 30,
    'max_depth': 5,
    'learning_rate': 0.05,
    'colsample_bytree': 0.9,
    'class_weight': 'balanced',
    'random_state': 42,
    'verbose': -1
}

model = LGBMClassifier(**best_params)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_val)
    y_pred_train = model.predict(X_train)
    
    # Metrikler (Val)
    val_acc = accuracy_score(y_val, y_pred_val)
    val_f1 = f1_score(y_val, y_pred_val, average='macro')
    val_prec = precision_score(y_val, y_pred_val, average='macro', zero_division=0)
    val_rec = recall_score(y_val, y_pred_val, average='macro', zero_division=0)
    val_bal_acc = balanced_accuracy_score(y_val, y_pred_val)
    
    # Metrikler (Train)
    train_acc = accuracy_score(y_train, y_pred_train)
    train_f1 = f1_score(y_train, y_pred_train, average='macro')
    train_prec = precision_score(y_train, y_pred_train, average='macro', zero_division=0)
    train_rec = recall_score(y_train, y_pred_train, average='macro', zero_division=0)
    train_bal_acc = balanced_accuracy_score(y_train, y_pred_train)
    
    cv_results.append({
        'Fold': str(fold),
        'Accuracy_Val': val_acc,
        'Macro_F1_Val': val_f1,
        'Precision_Macro_Val': val_prec,
        'Recall_Macro_Val': val_rec,
        'Balanced_Accuracy_Val': val_bal_acc,
        'Accuracy_Train': train_acc,
        'Macro_F1_Train': train_f1,
        'Precision_Macro_Train': train_prec,
        'Recall_Macro_Train': train_rec,
        'Balanced_Accuracy_Train': train_bal_acc
    })

results_df = pd.DataFrame(cv_results)

# Ortalama ve Standart Sapma Hesaplama
cols_to_avg = [
    'Accuracy_Val', 'Macro_F1_Val', 'Precision_Macro_Val', 'Recall_Macro_Val', 'Balanced_Accuracy_Val',
    'Accuracy_Train', 'Macro_F1_Train', 'Precision_Macro_Train', 'Recall_Macro_Train', 'Balanced_Accuracy_Train'
]

mean_row = {'Fold': 'Mean'}
std_row = {'Fold': 'Std'}

for col in cols_to_avg:
    numeric_vals = results_df[col].astype(float)
    mean_row[col] = numeric_vals.mean()
    std_row[col] = numeric_vals.std()

results_df = pd.concat([results_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)
results_df.to_csv('../outputs/cross_validation_results.csv', index=False)
results_df

,Fold,Accuracy_Val,Macro_F1_Val,Precision_Macro_Val,Recall_Macro_Val,Balanced_Accuracy_Val,Accuracy_Train,Macro_F1_Train,Precision_Macro_Train,Recall_Macro_Train,Balanced_Accuracy_Train
0,1,0.820999,0.647951,0.654918,0.642688,0.642688,0.921252,0.908537,0.879208,0.945388,0.945388
1,2,0.812777,0.658174,0.648159,0.670936,0.670936,0.925522,0.912178,0.883349,0.948233,0.948233
2,3,0.820999,0.687154,0.699008,0.678486,0.678486,0.925996,0.911365,0.881907,0.948485,0.948485
3,4,0.811512,0.676787,0.670290,0.684322,0.684322,0.928210,0.916876,0.889907,0.949950,0.949950
4,5,0.814674,0.658414,0.663491,0.654189,0.654189,0.925838,0.917923,0.892527,0.948546,0.948546
5,Mean,0.816192,0.665696,0.667173,0.666124,0.666124,0.925364,0.913376,0.885380,0.948120,0.948120
6,Std,0.004530,0.015874,0.019675,0.017312,0.017312,0.002533,0.003931,0.005609,0.001669,0.001669


## Çapraz Doğrulama Bulgularının Değerlendirilmesi

Elde edilen 5-Fold Stratified Cross-Validation sonuçlarına göre modelimizin performansı şu şekildedir:

### 1. Performans Stabilitesi (Tutarlılık):
- Foldlar arasında doğrulama metrikleri (Accuracy, Macro F1, Recall Macro ve Balanced Accuracy) incelendiğinde son derece tutarlı değerler elde edilmiştir. Örneğin doğrulama setindeki **Macro F1** değerleri tüm foldlar genelinde dar bir aralıkta dağılmaktadır.
- Bu durum, modelin veri setinin rastgele alt bölmelerine karşı hassas olmadığını ve istikrarlı bir öğrenme gerçekleştirdiğini gösterir.

### 2. Standart Sapmaların Düşüklüğü:
- Hesaplanan standart sapmalar (Std) doğrulama setinde tüm metrikler için son derece düşüktür.
- Düşük standart sapma, modelin farklı veri örneklemlerinde performans dalgalanması yaşamadığını doğrulamaktadır.

### 3. Modelin Genellenebilirliği:
- Ortalama doğrulama dengeli doğruluğu (**Balanced Accuracy_Val Mean**) ve ortalama Macro F1 skoru, modelin daha önce hiç görmediği veri foldlarında da yüksek performansını koruduğunu göstermektedir.
- Bu kararlılık modelin klinik olarak genellenebilir olduğunu gösterir.

### 4. Overfitting (Aşırı Öğrenme) Analizi:
- Eğitim metrikleri (Train) ile Doğrulama metrikleri (Val) arasındaki fark incelendiğinde:
  - Eğitim setindeki ortalama doğruluk ve F1 değerleri, doğrulama setine göre bir miktar daha yüksektir. Bu durum her makine öğrenmesi modelinde normal karşılanan bir durumdur.
  - Ancak aradaki fark oldukça küçüktür (Örn: Train F1 ile Val F1 arasındaki fark makul sınırlar içindedir).
  - Hyperparameter optimizasyonunda uygulanan L1 düzenlileştirme (`reg_alpha=5.0`) ve maksimum derinlik kısıtı (`max_depth=5`), modelin eğitim verisine aşırı uyum sağlamasını (ezberlemesini) engellemiş ve overfitting belirtisi olmaksızın genellenebilir kalmasını başarmıştır.
